In [2]:
!pip install -q --upgrade --force-reinstall "scikit-learn==1.7.2" chromadb sentence-transformers

import chromadb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from chromadb.utils import embedding_functions

documents = [
    {
        "source": "ai_overview.txt",
        "text": """
Artificial Intelligence is the field of creating computer systems that can perform tasks that normally require human intelligence.
AI can be used for learning, reasoning, problem solving, understanding language, and recognizing images.
Machine learning is a part of AI that allows computers to learn patterns from data.
Deep learning is a type of machine learning that uses neural networks with multiple layers.
"""
    },
    {
        "source": "embeddings.txt",
        "text": """
Embeddings convert text into numerical vectors.
These vectors represent the meaning and relationships between words, sentences, or documents.
Texts with similar meanings usually have vectors that are close to each other.
Sentence Transformers can be used to generate useful text embeddings.
Cosine similarity can be used to measure the similarity between two embedding vectors.
"""
    },
    {
        "source": "hybrid_search.txt",
        "text": """
Hybrid search combines semantic search and keyword search.
Semantic search finds information based on meaning using embeddings.
Keyword search finds information based on exact words using techniques such as TF-IDF.
Combining both methods can improve retrieval because semantic search understands meaning while keyword search can identify exact terms.
Hybrid search is useful in RAG systems because it can reduce empty or incomplete retrieval results.
"""
    }
]

chunks = []
metadatas = []
ids = []

for document in documents:
    parts = [
        part.strip()
        for part in document["text"].split("\n\n")
        if part.strip()
    ]

    for index, part in enumerate(parts):
        chunks.append(part)
        metadatas.append({
            "source": document["source"],
            "chunk_index": index
        })
        ids.append(f"{document['source']}_chunk_{index}")

embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

client = chromadb.Client()

try:
    client.delete_collection("hybrid_rag")
except:
    pass

collection = client.create_collection(
    name="hybrid_rag",
    embedding_function=embedding_function,
    metadata={"hnsw:space": "cosine"}
)

collection.add(
    documents=chunks,
    metadatas=metadatas,
    ids=ids
)

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

tfidf_matrix = vectorizer.fit_transform(chunks)

def semantic_search(query, top_k=5):
    results = collection.query(
        query_texts=[query],
        n_results=min(top_k, len(chunks)),
        include=["documents", "metadatas", "distances"]
    )

    output = []

    for document, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        similarity = max(0, 1 - distance)

        output.append({
            "document": document,
            "source": metadata["source"],
            "chunk_index": metadata["chunk_index"],
            "semantic_score": similarity
        })

    return output

def keyword_search(query, top_k=5):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, tfidf_matrix)[0]

    ranked_indices = scores.argsort()[::-1][:top_k]

    output = []

    for index in ranked_indices:
        if scores[index] > 0:
            output.append({
                "document": chunks[index],
                "source": metadatas[index]["source"],
                "chunk_index": metadatas[index]["chunk_index"],
                "keyword_score": float(scores[index])
            })

    return output

def hybrid_search(
    query,
    top_k=5,
    semantic_weight=0.6,
    keyword_weight=0.4,
    threshold=0.20
):
    semantic_results = semantic_search(query, top_k)
    keyword_results = keyword_search(query, top_k)

    combined = {}

    for result in semantic_results:
        key = (
            result["source"],
            result["chunk_index"]
        )

        combined[key] = {
            "document": result["document"],
            "source": result["source"],
            "chunk_index": result["chunk_index"],
            "semantic_score": result["semantic_score"],
            "keyword_score": 0.0
        }

    for result in keyword_results:
        key = (
            result["source"],
            result["chunk_index"]
        )

        if key not in combined:
            combined[key] = {
                "document": result["document"],
                "source": result["source"],
                "chunk_index": result["chunk_index"],
                "semantic_score": 0.0,
                "keyword_score": result["keyword_score"]
            }
        else:
            combined[key]["keyword_score"] = result["keyword_score"]

    for result in combined.values():
        result["hybrid_score"] = (
            semantic_weight * result["semantic_score"]
            + keyword_weight * result["keyword_score"]
        )

    ranked_results = sorted(
        combined.values(),
        key=lambda x: x["hybrid_score"],
        reverse=True
    )

    filtered_results = [
        result
        for result in ranked_results
        if result["hybrid_score"] >= threshold
    ]

    return filtered_results[:top_k]

def retrieve_context(query, top_k=3):
    results = hybrid_search(query, top_k)

    if not results:
        return {
            "found": False,
            "context": "",
            "results": []
        }

    context = "\n\n".join(
        result["document"]
        for result in results
    )

    return {
        "found": True,
        "context": context,
        "results": results
    }

def display_results(query):
    print("=" * 70)
    print("QUERY:", query)
    print("=" * 70)

    result = retrieve_context(query)

    if not result["found"]:
        print("No relevant information found.")
        print("The system could not find a sufficiently relevant document.")
        return

    print("\nRetrieved Results:\n")

    for index, item in enumerate(result["results"], 1):
        print(f"Result {index}")
        print("Source:", item["source"])
        print("Chunk:", item["chunk_index"])
        print("Semantic Score:", round(item["semantic_score"], 4))
        print("Keyword Score:", round(item["keyword_score"], 4))
        print("Hybrid Score:", round(item["hybrid_score"], 4))
        print("Text:", item["document"])
        print("-" * 70)

    print("\nRetrieved Context:")
    print(result["context"])

print("Hybrid Search RAG is ready.")

display_results("What is RAG and why is retrieval important?")

display_results("How do embeddings represent meaning?")

display_results("Why should we combine keyword search and semantic search?")

display_results("How does hybrid search improve retrieval?")

display_results("What is the capital of France?")

while True:
    query = input("\nEnter your question or type exit: ")

    if query.lower().strip() in ["exit", "quit", "bye"]:
        print("Exiting Hybrid Search RAG.")
        break

    result = retrieve_context(query)

    if not result["found"]:
        print("\nNo relevant information found.")
        print("Try asking a question related to the available documents.")
    else:
        print("\nRetrieved Context:\n")
        print(result["context"])

        print("\nSources:")
        for item in result["results"]:
            print(
                item["source"],
                "| Hybrid Score:",
                round(item["hybrid_score"], 4)
            )

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.6/111.6 kB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 114.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3